<a href="https://colab.research.google.com/github/Alfred-Doryele/Customer-Churn-Prediction/blob/main/notebooks/customer_churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 2.2.3
NumPy: 2.1.3


# Customer Churn Prediction — NexAfrica ML Internship
## 1. Business Understanding
What is customer churn?
Customer churn is when a customer cancels their subscription or service and leaves the company — for example, a telecom customer switching their phone or internet plan to a competitor.

Why does churn matter to a business?
Churn matters because acquiring a new customer is generally more expensive than retaining an existing one — through marketing, sales, and onboarding costs. High churn means the company is constantly spending money to replace customers it's losing, which hurts long-term profitability.

What is the goal of the prediction model?
The goal is to build a model that predicts, ahead of time, which currently active customers are at high risk of leaving — based on data like their contract type, tenure, monthly charges, payment method, and the services they use.

How does predicting churn help the company act?
If the company can identify at-risk customers before they leave, it can act proactively — for example, recommending a better-fit plan, reaching out with support, or offering a retention deal — instead of only reacting after the customer has already gone.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


In [3]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Shape (rows, columns): (7043, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [5]:
df['TotalCharges'].unique()[:20]

array(['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5',
       '1949.4', '301.9', '3046.05', '3487.95', '587.45', '326.8',
       '5681.1', '5036.3', '2686.05', '7895.15', '1022.95', '7382.25',
       '528.35', '1862.9'], dtype=object)

In [6]:
pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum()

np.int64(11)

In [7]:
df[pd.to_numeric(df['TotalCharges'], errors='coerce').isna()][['customerID', 'tenure', 'TotalCharges']]

,customerID,tenure,TotalCharges
488,4472-LVYGI,0,
753,3115-CZMZD,0,
936,5709-LVOEQ,0,
1082,4367-NUYAO,0,
1340,1371-DWPAZ,0,
3331,7644-OMVMY,0,
3826,3213-VVOLG,0,
4380,2520-SGTTA,0,
5218,2923-ARZLG,0,
6670,4075-WKNIU,0,


In [8]:
# Convert TotalCharges to numeric, turning bad values into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill the 11 missing values (tenure=0 customers) with 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Confirm the fix worked
print("Missing values now:", df['TotalCharges'].isna().sum())
print("New data type:", df['TotalCharges'].dtype)


Missing values now: 0
New data type: float64


In [9]:
# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check for duplicate customer IDs
print("Duplicate customer IDs:", df['customerID'].duplicated().sum())

# Check for missing values across all columns
print("\nMissing values per column:")
print(df.isna().sum().sum(), "total missing values")

Duplicate rows: 0
Duplicate customer IDs: 0

Missing values per column:
0 total missing values


In [10]:
# Distribution of churn
print(df['Churn'].value_counts())
print("\nPercentage:")
print(df['Churn'].value_counts(normalize=True) * 100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Percentage:
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [15]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df['customerID'].duplicated().sum())
print("Total missing values:", df.isna().sum().sum())

Duplicate rows: 0
Duplicate customer IDs: 0
Total missing values: 0


In [16]:
for col in df.select_dtypes(include='object').columns:
    print(col, ":", df[col].unique())
    print()

customerID : ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']

gender : ['Female' 'Male']

Partner : ['Yes' 'No']

Dependents : ['No' 'Yes']

PhoneService : ['No' 'Yes']

MultipleLines : ['No phone service' 'No' 'Yes']

InternetService : ['DSL' 'Fiber optic' 'No']

OnlineSecurity : ['No' 'Yes' 'No internet service']

OnlineBackup : ['Yes' 'No' 'No internet service']

DeviceProtection : ['No' 'Yes' 'No internet service']

TechSupport : ['No' 'Yes' 'No internet service']

StreamingTV : ['No' 'Yes' 'No internet service']

StreamingMovies : ['No' 'Yes' 'No internet service']

Contract : ['Month-to-month' 'One year' 'Two year']

PaperlessBilling : ['Yes' 'No']

PaymentMethod : ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Churn : ['No' 'Yes']



In [17]:
print(df['Churn'].value_counts())
print("\nPercentage:")
print(df['Churn'].value_counts(normalize=True) * 100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Percentage:
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [18]:
from sklearn.model_selection import train_test_split

X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

Training set size: (5634, 19)
Testing set size: (1409, 19)


In [19]:
df.to_csv('cleaned_churn_data.csv', index=False)
from google.colab import files
files.download('cleaned_churn_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ML Concepts

This is a classification problem, because the model predicts a category — whether a customer will churn (Yes) or not (No) — rather than a continuous number.

This is supervised learning, because the dataset already contains the correct answer for every customer (the Churn column), so the model learns by comparing its predictions against known outcomes.



Data Preparation Summary

Loaded the Telco Customer Churn dataset (7,043 customers, 21 columns) using Pandas
Found that TotalCharges was incorrectly stored as text instead of numbers, due to 11 new customers (tenure = 0) having a blank space instead of a value
Converted TotalCharges to numeric and filled those 11 missing values with 0, since new customers haven't been billed yet
Checked for duplicate rows and duplicate customer IDs, and confirmed there were none
Reviewed all categorical columns for inconsistent spelling or formatting
Split the dataset into training (80%) and testing (20%) sets, using stratified sampling to preserve the churn/no-churn ratio in both sets